# Extending Search Functions

This tutorial shows how to create custom search strategies by extending [`BaseSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/). We'll implement random and grid search as examples.

## 1. Imports

In [ ]:
import numpy as np
from alf_core.dataclasses import Candidate, Modality, TaskState
from alf_core.optimizer.search import BaseSearch

## 2. Define Custom Search Functions

Implement the `__call__()` method to generate candidate points. The method receives a [`TaskState`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/task_state/) and must return a list of [`Candidate`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/) objects with the appropriate [`Modality`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/).

In [ ]:
class RandomSearch(BaseSearch):
    """Random search in a bounded space."""

    def __init__(self, bounds: tuple[float, float], num_samples: int = 100):
        self.bounds = bounds
        self.num_samples = num_samples

    def __call__(self, task_state: TaskState, **kwargs) -> list[Candidate]:
        """Generate random candidates within bounds."""
        # Sample uniformly in the search space
        samples = np.random.uniform(low=self.bounds[0], high=self.bounds[1], size=self.num_samples)

        # Convert to Candidate objects
        candidates = [Candidate(data=x, modality=Modality.TABULAR) for x in samples]
        return candidates


class GridSearch(BaseSearch):
    """Grid search over a discrete set of points."""

    def __init__(self, bounds: tuple[float, float], num_points: int = 50):
        self.bounds = bounds
        self.num_points = num_points

    def __call__(self, task_state: TaskState, **kwargs) -> list[Candidate]:
        """Generate evenly-spaced grid of candidates."""
        # Create grid points
        grid = np.linspace(self.bounds[0], self.bounds[1], self.num_points)

        # Convert to Candidate objects
        candidates = [Candidate(data=x, modality=Modality.TABULAR) for x in grid]
        return candidates

    def get_metrics(self, task_state: TaskState) -> dict[str, float]:
        """Optionally return metrics about the search."""
        return {"grid_spacing": (self.bounds[1] - self.bounds[0]) / (self.num_points - 1)}

## 3. Usage Example

In [ ]:
# Create a mock task state (normally provided by the framework)
from alf_core.dataclasses import LabelledCandidates

# Initialize with some dummy data
initial_candidates = [Candidate(data=1.0, modality=Modality.TABULAR)]
initial_labels = np.array([0.5])
dataset = LabelledCandidates(candidates=initial_candidates, labels=initial_labels)

task_state = TaskState(
    dataset=dataset,
    surrogate=None,  # Not needed for search
    round=0,
    acq_batch_size=10,
)

# Random search
random_search = RandomSearch(bounds=(0.0, 10.0), num_samples=50)
random_candidates = random_search(task_state)
print(f"Random search generated {len(random_candidates)} candidates")
print(f"Sample values: {[c.data for c in random_candidates[:5]]}")

# Grid search
grid_search = GridSearch(bounds=(0.0, 10.0), num_points=20)
grid_candidates = grid_search(task_state)
print(f"\nGrid search generated {len(grid_candidates)} candidates")
print(f"First 5 values: {[c.data for c in grid_candidates[:5]]}")
print(f"Metrics: {grid_search.get_metrics(task_state)}")

## Key Points

- **Required method**: `__call__(task_state, **kwargs)` must return a list of `Candidate` objects
- **TaskState**: Contains current dataset, surrogate, round number, and batch size
- **Search space**: Define bounds, dimensions, or discrete sets based on your problem
- **Return type**: Always return `list[Candidate]` with appropriate modality
- **Optional**: Override `get_metrics()` to return search-specific metrics
- **Integration**: Search results are passed to acquisition functions for scoring
- **Advanced**: Can use `task_state.dataset` to avoid proposing previously seen points

Common search strategies:
- **Random**: Uniform or adaptive sampling
- **Grid**: Systematic exploration
- **Generator-based**: Use generative model (see `GeneratorSearch` in the framework)
- **Protocol-based**: Custom search protocols (see `ProtocolSearch`)